# YOLO12s + GhostConv + EMA-32 — verified pretrained transfer

**Varian ini memakai direct EMA-32 (non-residual).**

Notebook Kaggle ini memakai `yolo12s.pt` resmi sebagai sumber bobot. Bobot dengan bentuk yang kompatibel dipindahkan ke arsitektur GhostConv + direct EMA-32. EMA langsung mengalikan feature map dengan bobot attention; tidak ada residual-scale adapter. Inisialisasi GhostConv bersifat parsial, sehingga output awal tidak diklaim identik dengan YOLO12s asli. Setiap parameter model target kemudian dibuat trainable, trainer dipaksa memakai objek yang telah ditransfer tersebut, dan gradien aktual diperiksa sebelum optimizer pertama berjalan.

Catatan: `0 gradients` saat sebuah file `.pt` dibuka untuk inferensi adalah status parameter `requires_grad=False` dari checkpoint yang sudah di-strip oleh Ultralytics. Itu bukan mAP dan bukan nilai gradien backprop.

In [ ]:
# 1. Clone branch eksperimen dan install kode yang dimodifikasi. Aktifkan GPU dan Internet di Kaggle.
import json
import platform
import re
import subprocess
import sys
import zipfile
from pathlib import Path

WORKDIR = Path('/kaggle/working')
REPO_URL = 'https://github.com/danial2015/yolo-aceh-rdd2022.git'
REPO_BRANCH = 'yolo12-ghost-direct-ema32'
REPO_DIR = WORKDIR / 'yolo-aceh-rdd2022'

def log_section(title: str) -> None:
    print(f'\n{"=" * 88}\n{title}\n{"=" * 88}')

log_section('CLONE AND INSTALL MODIFIED REPOSITORY')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--depth', '1', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
REPO_COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
REPO_METADATA = WORKDIR / 'repository_revision.txt'
REPO_METADATA.write_text(f'repository={REPO_URL}\nbranch={REPO_BRANCH}\ncommit={REPO_COMMIT}\n', encoding='utf-8')
sys.path.insert(0, str(REPO_DIR))

import torch
import ultralytics
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
log_section('ENVIRONMENT')
print(f'Python      : {platform.python_version()}')
print(f'PyTorch     : {torch.__version__}')
print(f'Ultralytics : {ultralytics.__version__}')
print(f'Commit      : {REPO_COMMIT}')
print(f'CUDA ready  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name(0)}')


In [ ]:
# 2. Dataset dan hyperparameter. Samakan dengan baseline agar perbandingan mAP adil.
DATA_ROOT = Path('/kaggle/input/datasets/danialalfayyadh/ch-rdd-2022/datasets-china-split-fix')
DATA_YAML = WORKDIR / 'ch_rdd2022.yaml'
MODEL_YAML = REPO_DIR / 'ultralytics/cfg/models/12/yolo12-ghost-ema32.yaml'
CUSTOM_SOURCE_FILES = (
    REPO_DIR / 'ultralytics/nn/modules/conv.py', REPO_DIR / 'ultralytics/nn/modules/__init__.py',
    REPO_DIR / 'ultralytics/nn/tasks.py', MODEL_YAML,
)
EPOCHS, IMGSZ, BATCH, NBS = 160, 640, 16, 64
OPTIMIZER, LR0, MOMENTUM, WEIGHT_DECAY = 'SGD', 0.01, 0.937, 0.0005
PATIENCE, WORKERS, SEED, EMA_FACTOR = 0, 2, 42, 32
EXPERIMENT_NAME = 'yolo12s_ghost_direct_ema32_pretrained_ch_rdd2022'
RUNS_DIR = WORKDIR / 'runs'

DATA_YAML.write_text(f'''path: {DATA_ROOT}
train: train/images
val: val/images
test: test/images

nc: 5
names:
  0: D00
  1: D10
  2: D20
  3: D40
  4: Repair
''', encoding='utf-8')
assert DATA_ROOT.exists(), f'Dataset path tidak ditemukan: {DATA_ROOT}'
assert MODEL_YAML.exists() and all(path.exists() for path in CUSTOM_SOURCE_FILES)


In [ ]:
# 3. Bangun model 5 kelas, transfer bobot resmi yolo12s.pt, lalu verifikasi transfer.
from ultralytics import YOLO
from ultralytics.nn.modules import Conv, EMAAttention, GhostConv
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils.torch_utils import get_num_gradients, get_num_params

PRETRAINED_WEIGHTS = 'yolo12s.pt'
GHOST_SOURCE_TARGET_PAIRS = ((5, 6), (7, 8))

def target_layer_index(source_index: int) -> int:
    # EMA berada pada indeks 5 di model target; seluruh layer sumber setelah P3 digeser +1.
    return source_index + int(source_index >= 5)

def remap_yolo12s_weights(source_state: dict, target_state: dict) -> dict:
    transferred = {}
    pattern = re.compile(r'^model\.(\d+)(\..+)$')
    for source_key, source_tensor in source_state.items():
        match = pattern.match(source_key)
        if match is None:
            continue
        target_key = f'model.{target_layer_index(int(match.group(1)))}{match.group(2)}'
        if target_key in target_state and target_state[target_key].shape == source_tensor.shape:
            transferred[target_key] = source_tensor
    return transferred

def copy_bn_prefix(source_bn, target_bn, count: int) -> None:
    for name in ('weight', 'bias', 'running_mean', 'running_var', 'num_batches_tracked'):
        source_value, target_value = getattr(source_bn, name), getattr(target_bn, name)
        target_value.copy_(source_value if source_value.ndim == 0 else source_value[:count])

def initialize_ghost_from_pretrained_conv(source_conv: Conv, target_ghost: GhostConv) -> dict:
    primary_channels = target_ghost.cv1.conv.out_channels
    if source_conv.conv.weight.shape[1:] != target_ghost.cv1.conv.weight.shape[1:]:
        raise ValueError('Source Conv and GhostConv primary branch are incompatible.')
    with torch.no_grad():
        # Primary branch menerima separuh filter dan statistik BN asli. Cheap branch deterministik,
        # tetapi tetap memiliki BN+SiLU sehingga tidak dapat mereproduksi separuh filter sumber lainnya.
        target_ghost.cv1.conv.weight.copy_(source_conv.conv.weight[:primary_channels])
        copy_bn_prefix(source_conv.bn, target_ghost.cv1.bn, primary_channels)
        target_ghost.cv2.conv.weight.zero_()
        center_h, center_w = (size // 2 for size in target_ghost.cv2.conv.weight.shape[-2:])
        target_ghost.cv2.conv.weight[:, 0, center_h, center_w] = 1.0
        target_ghost.cv2.bn.weight.fill_(1.0)
        target_ghost.cv2.bn.bias.zero_()
        target_ghost.cv2.bn.running_mean.zero_()
        target_ghost.cv2.bn.running_var.fill_(1.0)
        target_ghost.cv2.bn.num_batches_tracked.zero_()
    return {'primary_channels_copied': primary_channels, 'cheap_branch': 'deterministic depthwise initialization; not source-equivalent'}

log_section('BUILD TARGET AND TRANSFER OFFICIAL YOLO12S WEIGHTS')
# YOLO facade dipakai untuk train/val, tetapi target harus dibangun sebagai 5 kelas sejak awal.
model = YOLO(str(MODEL_YAML), task='detect')
target_model = DetectionModel(str(MODEL_YAML), nc=5, verbose=False)
model.model = target_model
source_model = YOLO(PRETRAINED_WEIGHTS).model.float()
assert source_model.model[-1].nc == 80, 'Checkpoint sumber harus merupakan YOLO12s 80 kelas.'
assert target_model.model[-1].nc == 5, 'Head target wajib dibangun untuk 5 kelas RDD2022.'
assert target_model.model[-1].no == 69, 'Detect head 5 kelas harus memiliki 64 regresi + 5 klasifikasi.'
source_checkpoint_trainable = get_num_gradients(source_model)
target_state = target_model.state_dict()
transferred_state = remap_yolo12s_weights(source_model.state_dict(), target_state)
incompatible = target_model.load_state_dict(transferred_state, strict=False)

GHOST_INITIALIZATION = []
for source_index, target_index in GHOST_SOURCE_TARGET_PAIRS:
    source_layer, target_layer = source_model.model[source_index], target_model.model[target_index]
    if not isinstance(source_layer, Conv) or not isinstance(target_layer, GhostConv):
        raise TypeError(f'Expected Conv -> GhostConv at source {source_index}, target {target_index}.')
    GHOST_INITIALIZATION.append({
        'source_layer': source_index, 'target_layer': target_index,
        **initialize_ghost_from_pretrained_conv(source_layer, target_layer),
    })

# Checkpoint resmi disimpan untuk inferensi, sehingga parameter sumber bernilai requires_grad=False.
# Target harus trainable untuk fine-tuning; nilai bobotnya tidak berubah oleh operasi ini.
target_model.train()
for name, parameter in target_model.named_parameters():
    parameter.requires_grad_(not '.dfl.' in name)

# DFL memang bukan parameter yang dioptimasi oleh YOLO; semua parameter lain wajib trainable.
trainable_parameters = get_num_gradients(target_model)
total_parameters = get_num_params(target_model)
assert trainable_parameters > 0, 'Tidak ada parameter trainable setelah transfer.'
assert all(p.requires_grad for name, p in target_model.named_parameters() if '.dfl.' not in name)

# Pastikan semua tensor biasa yang kompatibel sama persis dengan sumber pretrained.
target_after_transfer = target_model.state_dict()
assert all(torch.equal(target_after_transfer[key], value) for key, value in transferred_state.items())
assert torch.equal(target_model.model[6].cv1.conv.weight, source_model.model[5].conv.weight[:target_model.model[6].cv1.conv.out_channels])
assert torch.equal(target_model.model[8].cv1.conv.weight, source_model.model[7].conv.weight[:target_model.model[8].cv1.conv.out_channels])

ema_layers = [(layer.i, layer.groups) for layer in target_model.model if isinstance(layer, EMAAttention)]
ghost_layers = [layer.i for layer in target_model.model if isinstance(layer, GhostConv)]
assert ema_layers == [(5, EMA_FACTOR)] and ghost_layers == [6, 8]
PRETRAINED_REPORT = {
    'source_weights': PRETRAINED_WEIGHTS,
    'source_nc': source_model.model[-1].nc,
    'target_nc': target_model.model[-1].nc,
    'pretrained_policy': 'compatible tensor remap plus partial GhostConv initialization; exact source-output continuity is not expected',
    'source_checkpoint_trainable_parameters_before_transfer': source_checkpoint_trainable,
    'standard_pretrained_tensors_transferred': len(transferred_state),
    'target_tensors': len(target_state),
    'remaining_uninitialized_tensors': list(incompatible.missing_keys),
    'target_parameters': total_parameters,
    'target_trainable_parameters_after_transfer': trainable_parameters,
    'ghost_initialization': GHOST_INITIALIZATION,
    'ema_mode': 'direct',
    'ema_is_new_and_trainable': True,
}
print(json.dumps(PRETRAINED_REPORT, indent=2))
target_model.info(detailed=False, verbose=True, imgsz=IMGSZ)
del source_model, target_state, transferred_state, target_after_transfer
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# 4. Trainer khusus ini menggunakan objek target yang SUDAH ditransfer, bukan membangun ulang model kosong.
from ultralytics.models.yolo.detect.train import DetectionTrainer
from ultralytics.utils.torch_utils import unwrap_model

TRANSFERRED_MODEL_ID = id(target_model)
GRADIENT_REPORT = {}

class VerifiedPretrainedTrainer(DetectionTrainer):
    initialized_model = None

    def get_model(self, cfg=None, weights=None, verbose=True):
        if self.initialized_model is None:
            raise RuntimeError('Model hasil transfer belum diberikan ke trainer.')
        initialized = self.initialized_model
        initialized.train()
        for name, parameter in initialized.named_parameters():
            parameter.requires_grad_(not '.dfl.' in name)
        return initialized

    def optimizer_step(self):
        # Dipanggil setelah backward dan sebelum optimizer.zero_grad(): inilah gradien aktual.
        if not GRADIENT_REPORT:
            named_parameters = dict(unwrap_model(self.model).named_parameters())
            probe_names = (
                'model.0.conv.weight',          # backbone pretrained
                'model.5.conv1x1.weight',       # EMA baru
                'model.6.cv1.conv.weight',      # Ghost primary dari pretrained
                'model.6.cv2.conv.weight',      # Ghost cheap branch
                'model.8.cv1.conv.weight',      # Ghost P5 primary dari pretrained
            )
            for name in probe_names:
                gradient = named_parameters[name].grad
                GRADIENT_REPORT[name] = {
                    'present': gradient is not None,
                    'finite': bool(gradient is not None and torch.isfinite(gradient).all()),
                    'l1_norm': float(gradient.detach().abs().sum()) if gradient is not None else 0.0,
                }
            if not all(item['present'] and item['finite'] and item['l1_norm'] > 0 for item in GRADIENT_REPORT.values()):
                raise RuntimeError(f'Gradient check failed: {json.dumps(GRADIENT_REPORT, indent=2)}')
            print('VERIFIED FIRST-STEP GRADIENTS:')
            print(json.dumps(GRADIENT_REPORT, indent=2))
        return super().optimizer_step()

def verified_trainer_factory(overrides, _callbacks):
    trainer = VerifiedPretrainedTrainer(overrides=overrides, _callbacks=_callbacks)
    trainer.initialized_model = target_model
    return trainer

def assert_trainer_uses_transferred_model(trainer):
    actual_model = unwrap_model(trainer.model)
    assert id(actual_model) == TRANSFERRED_MODEL_ID, 'Trainer membangun model lain; transfer pretrained tidak dipakai.'
    assert get_num_gradients(actual_model) > 0, 'Model trainer tidak memiliki parameter trainable.'
    assert actual_model.model[-1].nc == 5 and actual_model.model[-1].no == 69, 'Trainer tidak memakai detection head 5 kelas.'
    print(f'TRAINER VERIFIED: {get_num_gradients(actual_model):,} trainable parameters, object id preserved.')

model.add_callback('on_pretrain_routine_end', assert_trainer_uses_transferred_model)
log_section('TRAINING STARTED — VERIFIED PRETRAINED YOLO12S + GHOSTCONV + EMA-32')
print(f'epochs={EPOCHS}, imgsz={IMGSZ}, batch={BATCH}, nbs={NBS}, optimizer={OPTIMIZER}, lr0={LR0}, seed={SEED}')
model.train(
    trainer=verified_trainer_factory, data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, nbs=NBS,
    device=DEVICE, workers=WORKERS, project=str(RUNS_DIR), name=EXPERIMENT_NAME, exist_ok=True,
    pretrained=False, optimizer=OPTIMIZER, lr0=LR0, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY,
    cos_lr=False, patience=PATIENCE, seed=SEED, plots=True, verbose=True,
)
assert GRADIENT_REPORT, 'Optimizer belum berjalan; gradient report belum tersedia.'
RUN_DIR, BEST_PT, LAST_PT = Path(model.trainer.save_dir), Path(model.trainer.best), Path(model.trainer.last)
print(f'Run directory: {RUN_DIR}')
print(f'Best weights : {BEST_PT}')


In [ ]:
# 5. Evaluasi best.pt dan simpan seluruh artefak sebagai ZIP Kaggle.
log_section('BEST CHECKPOINT EVALUATION')
best_model = YOLO(str(BEST_PT))

def metric_summary(metrics) -> dict:
    return {
        'precision': float(metrics.box.mp), 'recall': float(metrics.box.mr),
        'map50': float(metrics.box.map50), 'map50_95': float(metrics.box.map),
        'save_dir': str(metrics.save_dir),
    }

val_metrics = best_model.val(data=str(DATA_YAML), split='val', imgsz=IMGSZ, batch=BATCH, device=DEVICE,
                             project=str(RUNS_DIR), name=f'{EXPERIMENT_NAME}_val', exist_ok=True, plots=True)
EVALUATION_REPORT = {'validation': metric_summary(val_metrics), 'first_optimizer_step_gradients': GRADIENT_REPORT}
test_label_dir = DATA_ROOT / 'test' / 'labels'
if test_label_dir.exists() and any(test_label_dir.glob('*.txt')):
    test_metrics = best_model.val(data=str(DATA_YAML), split='test', imgsz=IMGSZ, batch=BATCH, device=DEVICE,
                                  project=str(RUNS_DIR), name=f'{EXPERIMENT_NAME}_test', exist_ok=True, plots=True)
    EVALUATION_REPORT['test'] = metric_summary(test_metrics)
    TEST_OUTPUT_DIR = Path(test_metrics.save_dir)
else:
    predictions = best_model.predict(source=str(DATA_ROOT / 'test' / 'images'), imgsz=IMGSZ, device=DEVICE,
                                   conf=0.25, save=True, save_txt=True, project=str(RUNS_DIR),
                                   name=f'{EXPERIMENT_NAME}_test_predictions', exist_ok=True)
    TEST_OUTPUT_DIR = Path(predictions[0].save_dir) if predictions else RUNS_DIR
    EVALUATION_REPORT['test'] = {'status': 'labels unavailable; prediction only', 'save_dir': str(TEST_OUTPUT_DIR)}

EVALUATION_JSON = WORKDIR / f'{EXPERIMENT_NAME}_evaluation_metrics.json'
EVALUATION_JSON.write_text(json.dumps(EVALUATION_REPORT, indent=2), encoding='utf-8')
RUN_CONFIG = WORKDIR / f'{EXPERIMENT_NAME}_config.json'
RUN_CONFIG.write_text(json.dumps({
    'dataset_root': str(DATA_ROOT), 'repository_url': REPO_URL, 'repository_branch': REPO_BRANCH,
    'repository_commit': REPO_COMMIT, 'model_yaml': str(MODEL_YAML), 'ema_factor': EMA_FACTOR, 'ema_mode': 'direct',
    'pretrained_transfer': PRETRAINED_REPORT, 'first_optimizer_step_gradients': GRADIENT_REPORT,
    'epochs': EPOCHS, 'imgsz': IMGSZ, 'batch': BATCH, 'nbs': NBS, 'optimizer': OPTIMIZER,
    'lr0': LR0, 'momentum': MOMENTUM, 'weight_decay': WEIGHT_DECAY, 'seed': SEED,
    'best_checkpoint': str(BEST_PT), 'last_checkpoint': str(LAST_PT),
}, indent=2), encoding='utf-8')

ZIP_PATH = WORKDIR / f'{EXPERIMENT_NAME}_results.zip'
def add_to_zip(archive: zipfile.ZipFile, path: Path) -> int:
    if not path.exists():
        return 0
    files = [path] if path.is_file() else [item for item in path.rglob('*') if item.is_file()]
    for file_path in files:
        archive.write(file_path, file_path.relative_to(WORKDIR))
    return len(files)

with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    count = sum(add_to_zip(archive, Path(item)) for item in (
        RUN_DIR, Path(val_metrics.save_dir), TEST_OUTPUT_DIR, DATA_YAML, *CUSTOM_SOURCE_FILES,
        REPO_METADATA, RUN_CONFIG, EVALUATION_JSON,
    ))
print(json.dumps(EVALUATION_REPORT, indent=2))
print(f'ZIP created : {ZIP_PATH} ({count} files)')
from IPython.display import FileLink, display
display(FileLink(ZIP_PATH))
